<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/2026_RAG_step_by_step_gptApi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai --quiet
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

# Classification with raw OpenAI calls

In [2]:
from openai import OpenAI
import json
import textwrap

system_command="Is this sentiment positive or negative?"
user_input="I am feeling awful."

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.responses.create(model="gpt-5.4-nano",input=user_input,instructions=system_command,reasoning={"effort": "low"})

classification = response.output_text

print(textwrap.fill(f"{classification} \nInput was: {user_input}",width=100,replace_whitespace=False))

Negative. 
Input was: I am feeling awful.


# Question-Answer with raw OpenAI calls

In [3]:
from openai import OpenAI
import json

system_command="Please answer this question from the user. Keep the answer short and concise."

def answer_question(user_input):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  response = client.responses.create(model="gpt-5.4-nano",input=user_input,instructions=system_command,reasoning={"effort": "low"})

  msg = response.output_text
  return msg

In [6]:
answer_question("What is the capital of France?")

'The capital of France is **Paris**.'

In [7]:
import textwrap
answer=answer_question("What did Trane announce in March 2026?")
print(textwrap.fill(answer,width=100))

I don’t have access to Trane’s March 2026 announcements, and my knowledge base may not include
events that far ahead.    If you share the link/headline (or the text of the announcement), I can
summarize exactly what Trane announced.


# Augmented Question-Answer

In [8]:
def augmentation_data():
  new_information="""
  Trane Technologies Optimizes Industry‑First Thermal Management Reference Design for AI Factories, Introduces Two New Designs
Mar 16, 2026
Company improves design efficiency by nearly 10% and launches two additional Trane® Continuum Rubin DSX reference designs

SWORDS, Ireland--(BUSINESS WIRE)-- Trane Technologies (NYSE:TT), a global climate innovator, today announced major enhancements to its industry-first comprehensive thermal management reference design for gigawatt-scale AI factories and unveiled two Trane Continuum Rubin DSX reference designs.

Engineered specifically to integrate with the NVIDIA® Omniverse™ DSX Blueprint for AI data centers, the new system optimizations achieve a nearly 10% improvement in overall thermal management performance compared to the original 1-gigawatt reference design announced in October. This frees up 22 megawatts of cooling capacity that can be redirected to IT power, helping boost compute output without increasing total energy consumption.

“Since the launch of our industry-first thermal management system reference design, our team has continued to work hand-in-hand with NVIDIA to push the boundaries of efficiency, scalability and intelligent simulation for gigawatt-scale AI infrastructure,” said Mauro Atalla, Senior Vice President and Chief Technology and Sustainability Officer, Trane Technologies. “These latest advancements represent a major step forward in helping enable the world’s most demanding AI and high-performance computing environments to scale sustainably, reliably and with accelerated digital insight.”

“Efficiently scaling gigawatt-scale AI factories requires a fundamental shift in how we approach thermal management and data center infrastructure simulation,” said Vladimir Troy, Vice President of AI Infrastructure at NVIDIA. “Trane Technologies’ integration with the NVIDIA Omniverse DSX blueprint enables the creation of high-fidelity digital twins that help operators optimize cooling performance and maximize energy efficiency for the next generation of AI workloads.”

Through continued collaboration with NVIDIA, including as an NVIDIA Partner Network member, Trane Technologies has also expanded its Trane Continuum Rubin DSX reference design portfolio with two additional high-efficiency solutions for large-scale AI deployments:

250-Megawatt Duplex Simplified System Design: This new design, available now, supports extended free cooling use and incorporates integrated heat recovery, which helps reduce system complexity and results in a 14% increase in thermal management system efficiency with 10% of the heat rejection load going to heat recovery.
1-Gigawatt AI Factory Mag-Bearing Air-Cooled System Architecture: This new design, available soon, features a streamlined air-cooled approach using 3-megawatt units to help reduce equipment count and eliminate the need for integrated waterside economizers. The architecture incorporates Trane’s latest magnetic-bearing air-cooled chiller, providing oil-free operation, high efficiency and quieter, more efficient performance through Trane chiller plant controls.
Trane Technologies has also advanced its digital capabilities for adopting the Omniverse DSX Blueprint with more automated, scalable OpenUSD based SimReady assets. Enhanced with structured metadata, these assets will support upcoming reference design updates, helping improve configurability, accuracy and readiness for high‑scale digital‑twin and AI‑driven simulation workflows.

About Trane Technologies

Trane Technologies is a global climate innovator. Through our strategic brands Trane® and Thermo King®, and our portfolio of environmentally responsible products and services, we bring efficient and sustainable climate solutions to buildings, homes and transportation. For more on Trane Technologies, visit www.tranetechnologies.com.

About Trane

Trane® – by Trane Technologies (NYSE: TT), a global climate innovator – creates comfortable, energy efficient indoor environments through a broad portfolio of heating, ventilating and air conditioning systems and controls, services, parts and supply. For more information, please visit www.trane.com or www.tranetechnologies.com.

"""
  return new_information

In [9]:
from openai import OpenAI
import json


system_command="Please answer this question from the user. Keep the answer short and concise."

def answer_question_augmented(user_input):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  response = client.responses.create(model="gpt-5.4-nano",input=
                                     [{'role':'user',"content":user_input},
                                      {'role':'assistant',"content":augmentation_data()}],
                                     instructions=system_command,reasoning={"effort": "low"})

  msg = response.output_text
  return msg

In [10]:
import textwrap
answer=answer_question_augmented("What did Trane announce in march 2026?")
print(textwrap.fill(answer,width=100))

In March 2026, Trane Technologies announced upgrades to its gigawatt-scale AI factory thermal
management reference design—improving overall thermal performance by nearly 10%—and also introduced
two new **Trane Continuum Rubin DSX** reference designs (a 250‑MW duplex simplified system design
and a 1‑GW mag-bearing air-cooled architecture).


# Retrieval-Augmented Generation: Storage

---



In [11]:
!pip install langchain faiss-cpu transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.3 MB/s eta 0:00:00


In [12]:
import faiss
import numpy as np
from openai import OpenAI

client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def embed_text_openai(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    embeddings = response.data[0].embedding
    return np.array(embeddings)

def chunk_text(text, chunk_size=1000, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def store_in_faiss(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

In [14]:
import torch
with open('/content/trane_file_G2.txt', 'r',  encoding='latin-1') as file:
    text = file.read()
chunks = chunk_text(text)
embeddings = np.vstack([embed_text_openai(chunk) for chunk in chunks])
index = store_in_faiss(embeddings)
faiss.write_index(index, "faiss_index.index")

print(f"Chunk count: {len(chunks)}")
for i in range(3):
  print(chunks[i]+"\n***********\n")

Chunk count: 12
WALL MOUNTED SPLIT-TYPE AIR CONDITIONERS Service Manual Models TRANE-09CO-FIXED-A1 TRANE-12CO-FIXED-A1 TRANE-18CO-FIXED-A1 TRANE-24CO-FIXED-A1 TRANE-09CH-FIXED-A1 TRANE-12CH-FIXED-A1 TRANE-18CH-FIXED-A1 TRANE-24CH-FIXED-A1 CONTENTS Part . Technical InformationÉÉÉÉÉÉÉÉÉÉÉÉÉ...É.ÉÉÉÉ 2 1. Important NoticeÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉ..ÉÉÉ..2 2. Production DimensionsÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉ.3 3. Refrigeration cycle diagramÉÉÉÉÉÉÉÉÉÉÉÉÉ..ÉÉÉÉÉ...4 4. Wiring DiagramÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉ..5 5. Electronic Controller IntroductionÉÉÉÉÉÉÉÉÉÉÉÉÉÉ.ÉÉ...9 PART . Installation and MaintenanceÉÉÉÉÉ..ÉÉ.ÉÉÉ.ÉÉÉÉ16 1. Notes for installation and maintenance.ÉÉÉÉÉÉ..ÉÉÉÉÉÉÉ..16 2. InstallationÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉ..ÉÉÉÉÉÉÉ.24 3. MaintenanceÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉ..ÉÉ.ÉÉ31 4. Exploded view and parts listÉÉÉÉÉÉÉÉÉÉ..ÉÉÉÉÉÉÉÉ..35 5. Disassembly IDU & ODU ÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉÉ.36 APPENDIX 1. The comparison table of CELSIUS-FAHRENHEIT temperatureÉÉ.......45 2. The Pipe length and Gas chargingÉÉÉÉÉÉÉÉÉÉÉÉÉ.ÉÉ

In [15]:
def retrieve_similar_chunks(query, index, chunks, top_k=3):
    query_embedding = embed_text_openai(query).reshape(1,-1)
    distances, indices = index.search(query_embedding, top_k)
    return [(chunks[idx], distances[0][i]) for i, idx in enumerate(indices[0])]

#query = input("Enter a query:")
query = "What precautions should I take while working with the unit?"
print(f"\n\nQuery: {query}")
top_k = 3  # Number of similar chunks to retrieve
similar_chunks = retrieve_similar_chunks(query, index, chunks, top_k)

for chunk, distance in similar_chunks:
  print(f"Distance: {distance} Chunk: {chunk}\n")



Query: What precautions should I take while working with the unit?
Distance: 0.9432034492492676 Chunk: compressor runs continuously more than 6 min, the counter of overheat stop-working protection will be reset to zero and start a new counting process. The failure and times for protection will eliminate immediately once the unit to be switched off, on fan mode or changed to be heating mode from others. 5.2.11 Complementary 5.2.11.1 Energy saving (ECO) Function effective on Cooling and Heating mode only. On cooling mode, the set temperature range from 26.(79.) to 31.(88.). on heating mode, from 16.(61.) to 25.(77.). 5.2.11.2 TURBO Function effective on Cooling, Heating, Fan and Auto modes, and fan speed operates on highest. 5.2.12 Calibration Test Mode: Within 3 min while indoor unit switch on, and set the unit as: 1) Cooling mode. 2) set temperature to 30.. 3) Mid-fan speed. by press ECO button 7 times within 8s, the unit will change to calibration test mode, and the buzzer sounds 3 

In [16]:
def customized_augmentation_data(input_question : str) -> str:
  #
  # Return information based on the input question asked.
  #
  query_embedding = embed_text_openai(query).reshape(1,-1)
  distances, indices = index.search(query_embedding, top_k)
  new_information="\n\n".join(chunks[idx] for i, idx in enumerate(indices[0]))
  print(f"Retrieval:\n\\n {new_information}\nn")
  return new_information

# RAG: With retrieval

In [17]:
from openai import OpenAI
import json


system_command="Please answer this question from the user. Keep the answer short and concise. Only use the provided information to answer the question."

def answer_question_augmented(user_input):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  response = client.responses.create(model="gpt-5.4-nano",input=
                                     [{'role':'user',"content":user_input},
                                      {'role':'assistant',"content":customized_augmentation_data(user_input)}],
                                     instructions=system_command,reasoning={"effort": "low"})

  msg = response.output_text
  return msg

In [20]:
import textwrap
answer=answer_question_augmented("What precautions should I take while working with the unit?")
print("\n*** ANSWER ***\n"+textwrap.fill(answer,width=100))

Retrieval:
\n compressor runs continuously more than 6 min, the counter of overheat stop-working protection will be reset to zero and start a new counting process. The failure and times for protection will eliminate immediately once the unit to be switched off, on fan mode or changed to be heating mode from others. 5.2.11 Complementary 5.2.11.1 Energy saving (ECO) Function effective on Cooling and Heating mode only. On cooling mode, the set temperature range from 26.(79.) to 31.(88.). on heating mode, from 16.(61.) to 25.(77.). 5.2.11.2 TURBO Function effective on Cooling, Heating, Fan and Auto modes, and fan speed operates on highest. 5.2.12 Calibration Test Mode: Within 3 min while indoor unit switch on, and set the unit as: 1) Cooling mode. 2) set temperature to 30.. 3) Mid-fan speed. by press ECO button 7 times within 8s, the unit will change to calibration test mode, and the buzzer sounds 3 times. PART . Installation and Maintenance 1. Notes for installation and maintenance Safety

# Extract data from PDF file

In [21]:
!pip install PyPDF2 --quiet

import PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.4 MB/s eta 0:00:00


In [ ]:
file_path = "/content/technical-pb-mini-split-fixo-60hz-4mcw15-sn-pb-07092021.pdf"

with open(file_path, 'rb') as file:
    pdf_reader = PyPDF2.PdfReader(file)
    text_content = []
    for page in pdf_reader.pages:
        text_content.append(page.extract_text())

full_text = "\n\n".join(text_content)

with open('trane_file_out.txt', 'w') as file:
    file.write(full_text)